# Path-signature features for realized volatility

The **signature** of a path is a hierarchy of iterated integrals that
describes the path's *geometry* — the order in which things happened, not
just where it ended up. It comes out of rough path theory (Lyons), and it
has one property that makes it interesting for volatility specifically:

> Apply the **lead-lag transform** to a price path and the antisymmetric
> part of its second-level signature is *exactly* the path's quadratic
> variation — realized variance.

So realized volatility isn't something you bolt onto signature features. It
already lives inside them, at level 2. Everything above level 2 describes
finer structure in how the price got there. The question this notebook asks
is whether that finer structure is worth anything for forecasting the *next*
ten minutes of volatility.

This uses [`sigtrade`](https://github.com/FilipNowakowicz/sigtrade), a small
scikit-learn-compatible library for causal, rolling-window signature
features on financial time series.

**On honesty:** the private leaderboard for this competition was rescored on
market data from after it closed, so nothing here can be turned into a
leaderboard position and nothing here tries. Every comparison below is
out-of-fold on the training data, with the competition's metric, with every
arm going into the same learner on the same folds.

In [ ]:
!pip install -q iisignature
!pip install -q "sigtrade @ git+https://github.com/FilipNowakowicz/sigtrade"

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sigtrade import lead_lag, log_signature, preprocess_path, signature

DATA = Path("/kaggle/input/optiver-realized-volatility-prediction")
targets = pd.read_csv(DATA / "train.csv")
targets.head()

## 1. The identity, checked on real data

Before building anything, let's confirm the claim. Take one stock's book,
one ten-minute segment, and the WAP path it traces. Compute the realized
variance directly. Then compute the depth-2 signature of the *lead-lag*
embedding of that same path and take `S^(1,2) - S^(2,1)`.

They should agree to floating-point precision — not approximately, exactly.

In [ ]:
book = pd.read_parquet(DATA / "book_train.parquet" / "stock_id=0")

def wap(df, level=1):
    bid_p, ask_p = df[f"bid_price{level}"], df[f"ask_price{level}"]
    bid_s, ask_s = df[f"bid_size{level}"], df[f"ask_size{level}"]
    return (bid_p * ask_s + ask_p * bid_s) / (bid_s + ask_s)

segment = book[book["time_id"] == book["time_id"].iloc[0]]
log_price = np.log(wap(segment).to_numpy())[:, None]

realized_variance = np.sum(np.diff(log_price[:, 0]) ** 2)

sig = signature(lead_lag(log_price), depth=2, backend="iisignature")
s12, s21 = sig[3], sig[4]

print(f"realized variance     {realized_variance:.12e}")
print(f"S(1,2) - S(2,1)       {s12 - s21:.12e}")
print(f"ratio                 {(s12 - s21) / realized_variance:.15f}")

That is the whole argument for using signatures here, in three lines. The
classical estimator is a *coordinate* of the feature vector. Anything the
model gains beyond the baseline has to come from the other coordinates.

## 2. Building features

For each `(stock_id, time_id)` segment we:

1. forward-fill the WAP onto a regular one-second grid — the signature is
   invariant under reparametrisation, so this changes no geometry; it only
   makes the added time channel measure real elapsed seconds;
2. take **suffix windows** of 600, 300 and 150 seconds — every window ends
   at the segment's close, so nothing is visible that a live predictor
   wouldn't have;
3. time-augment, lead-lag, and take the depth-3 **log-signature** of each.

The log-signature is the compressed form: it drops the coordinates that the
shuffle identities make redundant. For a 4-dimensional path at depth 3 that
is 30 numbers instead of 84.

In [ ]:
SECONDS = 600
WINDOWS = (600, 300, 150)
DEPTH = 3

def wap_grid(book, n_seconds=SECONDS):
    frame = pd.DataFrame({
        "time_id": book["time_id"].to_numpy(),
        "second": book["seconds_in_bucket"].to_numpy(),
        "wap": wap(book).to_numpy(),
    })
    grid = frame.pivot_table(index="time_id", columns="second", values="wap", aggfunc="last")
    return grid.reindex(columns=range(n_seconds)).ffill(axis=1).bfill(axis=1)

def signature_features(grid, depth=DEPTH, windows=WINDOWS):
    log_prices = np.log(grid.to_numpy())
    blocks = []
    for window in windows:
        suffix = log_prices[:, -window:]
        blocks.append(np.array([
            log_signature(
                preprocess_path((row - row[0])[:, None],
                                time_augmentation=True, lead_lag_transform=True),
                depth, backend="iisignature")
            for row in suffix
        ]))
    columns = [f"sig_w{w}_{i}" for w in windows for i in range(blocks[0].shape[1])]
    return pd.DataFrame(np.hstack(blocks), columns=columns, index=grid.index)

def har_features(grid, windows=(600, 300, 150, 60, 30)):
    returns = np.diff(np.log(grid.to_numpy()), axis=1)
    out = {}
    for window in windows:
        suffix = returns[:, -window:]
        rv = np.sqrt(np.square(suffix).sum(axis=1))
        out[f"har_rv_{window}"] = rv
        out[f"har_absmean_{window}"] = np.abs(suffix).mean(axis=1)
        out[f"har_active_{window}"] = (suffix != 0).sum(axis=1) / window
    return pd.DataFrame(out, index=grid.index)

### A stock subset

Twenty stocks, drawn from a seeded generator. Building signatures for all
112 is a lot of compute for a notebook, and the point here is the
comparison, not the scale. The subset is fixed so every arm sees identical
rows.

In [ ]:
rng = np.random.default_rng(0)
all_stocks = np.sort(targets["stock_id"].unique())
STOCKS = sorted(rng.choice(all_stocks, size=20, replace=False).tolist())

frames = []
for stock_id in STOCKS:
    book = pd.read_parquet(DATA / "book_train.parquet" / f"stock_id={stock_id}")
    grid = wap_grid(book)
    stock_targets = targets[targets["stock_id"] == stock_id].set_index("time_id")
    grid = grid.loc[[t for t in grid.index if t in stock_targets.index]]

    block = pd.concat([har_features(grid), signature_features(grid)], axis=1)
    block["stock_id"] = stock_id
    block["time_id"] = grid.index
    block["target"] = stock_targets.loc[grid.index, "target"].to_numpy()
    frames.append(block.reset_index(drop=True))

table = pd.concat(frames, ignore_index=True)
print(table.shape)
table.head()

## 3. Scoring, without leaking

Two things matter here and both are easy to get wrong.

**The metric.** RMSPE divides by the truth, so a miss on a quiet segment
counts as much as the same *relative* miss on a violent one. Training on
raw squared error would optimise something else entirely. Weighting each
sample by `1/y²` under squared error is algebraically the same objective as
RMSPE, so that's what we do.

**The folds.** A `time_id` is one instant of market time observed across
*every* stock, and volatility is strongly correlated across stocks at the
same instant. Letting one stock's row from a given `time_id` train a model
that is then scored on another stock's row from the same `time_id` hands
over most of the answer. `GroupKFold` on `time_id` closes that.

Note what this is *not*: a walk-forward split. The organisers deliberately
shuffled `time_id`s, so their order isn't chronological and a true temporal
split can't be reconstructed from the shipped data. That's a real limitation
of any CV on this dataset, and it's better stated than glossed over.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold

def rmspe(y_true, y_pred):
    return float(np.sqrt(np.mean(np.square((y_true - y_pred) / y_true))))

def run_arm(X, y, groups, n_splits=5):
    predictions = np.zeros(len(y))
    for train_idx, test_idx in GroupKFold(n_splits=n_splits).split(X, y, groups):
        model = HistGradientBoostingRegressor(
            max_iter=400, learning_rate=0.05, max_leaf_nodes=31,
            min_samples_leaf=50, l2_regularization=1.0,
            early_stopping=False, random_state=0)
        model.fit(X.iloc[train_idx], y[train_idx],
                  sample_weight=1.0 / np.square(y[train_idx]))
        predictions[test_idx] = model.predict(X.iloc[test_idx])
    return predictions

y = table["target"].to_numpy()
groups = table["time_id"].to_numpy()

arms = {
    "har": [c for c in table.columns if c.startswith("har_")],
    "sig": [c for c in table.columns if c.startswith("sig_")],
    "sig+har": [c for c in table.columns if c.startswith(("sig_", "har_"))],
}

results = {"naive (RV of observed window)": (1, rmspe(y, table["har_rv_600"].to_numpy()))}
predictions = {}
for name, columns in arms.items():
    X = table[columns + ["stock_id"]]
    predictions[name] = run_arm(X, y, groups)
    results[name] = (len(columns), rmspe(y, predictions[name]))
    print(f"{name:>10s}  {len(columns):4d} features  RMSPE {results[name][1]:.5f}")

pd.DataFrame(
    [{"arm": k, "features": v[0], "RMSPE": round(v[1], 5)} for k, v in results.items()]
)

## 4. Is the difference real?

RMSPE differences between arms are often smaller than the noise in either.
A bootstrap over whole `time_id` groups — not individual rows, which are
nowhere near independent — gives an interval on the improvement.

In [ ]:
def paired_bootstrap(y, baseline, challenger, groups, n_resamples=1000, seed=0):
    rng = np.random.default_rng(seed)
    unique = np.unique(groups)
    index_by_group = {g: np.flatnonzero(groups == g) for g in unique}
    deltas = np.empty(n_resamples)
    for i in range(n_resamples):
        drawn = rng.choice(unique, size=len(unique), replace=True)
        idx = np.concatenate([index_by_group[g] for g in drawn])
        deltas[i] = rmspe(y[idx], baseline[idx]) - rmspe(y[idx], challenger[idx])
    observed = rmspe(y, baseline) - rmspe(y, challenger)
    low, high = np.percentile(deltas, [2.5, 97.5])
    return observed, low, high, float(np.mean(deltas <= 0))

for baseline, challenger in [("har", "sig"), ("har", "sig+har")]:
    observed, low, high, p = paired_bootstrap(
        y, predictions[baseline], predictions[challenger], groups)
    print(f"{challenger} vs {baseline}: {observed:+.5f} RMSPE "
          f"({100 * observed / rmspe(y, predictions[baseline]):+.2f}%), "
          f"95% CI [{low:+.5f}, {high:+.5f}], p(no improvement) = {p:.3f}")

## What to take from this

Read the numbers your run actually produced, not a story decided in advance.
The three outcomes worth distinguishing:

- **`sig` alone beats `har` alone.** The signature family carries the
  classical estimator *and* something more.
- **`sig+har` beats `har`, but `sig` alone doesn't.** Signatures add
  information at the margin without being a replacement — the usual and most
  likely outcome for a feature family this general.
- **Neither beats `har`.** Then the extra levels are describing structure
  that doesn't forecast, and the honest conclusion is that on this task,
  depth beyond the quadratic-variation term isn't paying for itself.

Any of those is a result. The third is the one most often left out of
write-ups, which is exactly why it's worth reporting when it happens.

### Where to go next

`sigtrade`'s actual target isn't segmented data like this — it's *rolling*
features on a continuous stream, where sliding a window by one tick costs
O(1) once the window is full, using Chen's identity and the group inverse
of the departing segment instead of recomputing the whole window. That's
the part of the library where the algebra does load-bearing work. It buys
129× over the default backend at window 1200, though a compiled backend
still wins on short windows — `benchmarks/streaming/` has the crossover.

- Repository: <https://github.com/FilipNowakowicz/sigtrade>
- The math, worked from a live example: `docs/notes.html`
- This benchmark, in full: `benchmarks/orvp/`

If you try it on something and it doesn't work, that's the most useful
thing you could report.